# 01 — Action Unit explorationInteractive companion to the real-time detector.  Use it to **look at the signalsthe pipeline produces** before trusting any number it prints:1. run the pipeline over a recording (or a live camera) and keep every intermediate,2. plot the stress index and the per-AU intensities over time,3. plot the residual optical flow per region and mark the detected micro-expression bursts,4. check the temporal spectrum of each region — micro-expressions live in a band that   slow head drift does not reach,5. inspect the personal baseline that everything is measured against.> **Not a medical device.**  Nothing here is validated against a clinical outcome.> Treat the score as a research read-out, never as a diagnosis.

In [ ]:
import sysfrom pathlib import Pathsys.path.insert(0, str(Path.cwd().parent))import matplotlib.pyplot as pltimport numpy as npfrom src.camera import CameraStreamfrom src.optical_flow import MICRO_BAND_HZ, band_energy_ratiofrom src.pipeline import StressPipelinefrom src.au_estimator import AU_ORDER, STRESS_ACTION_UNITSplt.rcParams["figure.figsize"] = (11, 4)plt.rcParams["axes.grid"] = Trueplt.rcParams["grid.alpha"] = 0.3

## 1. Record a session`SOURCE` accepts exactly what `--camera` on the command line does: a USB deviceindex (`0`), an RTSP URL, or a path to a video file.  Working from a **recordedfile** is strongly recommended while exploring — it makes every run reproducible.Sit still with a relaxed, forward-facing expression for the first`CALIBRATION_SECONDS`: that is what the personal baseline is built from.

In [ ]:
SOURCE = 0                    # 0 = USB webcam | "rtsp://admin:@192.168.1.15:554/stream1" | "clip.mp4"MAX_FRAMES = 900              # ~30 s at 30 fpsCALIBRATION_SECONDS = 3.0records = []pipeline = StressPipeline(calibration_seconds=CALIBRATION_SECONDS)with CameraStream(SOURCE, flip=True) as stream:    for frame in stream.frames(max_frames=MAX_FRAMES):        result = pipeline.process(frame.image, frame.timestamp)        if result.au_frame is None:            continue        records.append({            "t": result.timestamp,            "score": result.stress.score,            "instant": result.stress.instant,            "calibrated": result.stress.calibrated,            "quality": result.au_frame.quality,            "micro_rate": result.au_frame.micro_rate,            "blink_rate": result.au_frame.blink_rate,            "masked_smile": result.au_frame.masked_smile,            "aus": {code: result.au_frame.intensity(code) for code in AU_ORDER},            "motion": {name: m.directed for name, m in result.motions.items()},        })pipeline.detector.close()t0 = records[0]["t"]t = np.array([r["t"] - t0 for r in records])print(f"{len(records)} frames over {t[-1]:.1f} s  ({len(records) / max(t[-1], 1e-6):.1f} fps)")print(f"MediaPipe backend: {pipeline.detector.backend}")

## 2. The stress index over timeThe smoothed index is the headline number; the instantaneous one shows how much ofit is momentary.  The shaded span is the calibration period, where no score isreported at all.

In [ ]:
score = np.array([r["score"] for r in records])instant = np.array([r["instant"] for r in records])calibrated = np.array([r["calibrated"] for r in records])fig, ax = plt.subplots()ax.plot(t, instant, lw=0.8, alpha=0.45, label="instantaneous")ax.plot(t, score, lw=2.0, label="smoothed index")if (~calibrated).any():    ax.axvspan(t[0], t[~calibrated][-1], color="grey", alpha=0.15, label="calibration")for level, name in ((2, "mild"), (4, "moderate"), (6, "elevated"), (8, "high")):    ax.axhline(level, color="k", lw=0.5, alpha=0.25)    ax.text(t[-1], level, f" {name}", va="center", fontsize=8, alpha=0.6)ax.set(xlabel="time [s]", ylabel="stress index", ylim=(0, 10), title="Stress index")ax.legend(loc="upper left")plt.tight_layout()

## 3. Which Action Units drive itRows are AUs, columns are frames.  This is the most useful plot for sanity-checking:a stress reading that is not backed by a plausible AU pattern is an artefact.Look specifically for **AU12 without AU6** — a smile that never reaches the eyes.

In [ ]:
matrix = np.array([[r["aus"][code] for r in records] for code in AU_ORDER])fig, ax = plt.subplots(figsize=(11, 4.5))image = ax.imshow(matrix, aspect="auto", origin="lower", vmin=0, vmax=5,                  cmap="magma", extent=[t[0], t[-1], -0.5, len(AU_ORDER) - 0.5])ax.set_yticks(range(len(AU_ORDER)))ax.set_yticklabels([f"{c}  {STRESS_ACTION_UNITS[c].name}" for c in AU_ORDER], fontsize=8)ax.set(xlabel="time [s]", title="Action Unit intensity (FACS 0-5)")fig.colorbar(image, ax=ax, label="intensity")plt.tight_layout()masked = np.array([r["masked_smile"] for r in records])print(f"masked-smile frames: {masked.sum()} / {len(masked)} ({100 * masked.mean():.1f} %)")

## 4. Residual motion and the detected leaks`directed` motion is `|flow| x coherence`: incoherent noise is suppressed, coherentmotion in one direction survives.  Dashed lines mark the bursts the detector acceptedas micro-expressions (40–500 ms).  The horizontal line is the absolute noise gate —without it, a subject sitting still produces enormous z-scores from pure flow noise.

In [ ]:
REGIONS = ["brow_inner_right", "brow_inner_left", "cheek_left", "lip_lower", "chin"]events = list(pipeline.estimator.micro_detector.events)gate = pipeline.estimator.micro_detector.min_motionfig, axes = plt.subplots(len(REGIONS), 1, figsize=(11, 1.7 * len(REGIONS)), sharex=True)for ax, region in zip(np.atleast_1d(axes), REGIONS):    ax.plot(t, [r["motion"].get(region, 0.0) for r in records], lw=0.9)    ax.axhline(gate, color="grey", ls=":", lw=1)    for event in events:        if event.region == region:            ax.axvline(event.apex - t0, color="crimson", ls="--", lw=1)    ax.set_ylabel(region, fontsize=7)np.atleast_1d(axes)[-1].set_xlabel("time [s]")np.atleast_1d(axes)[0].set_title("Residual coherent motion per region (px per frame pair)")plt.tight_layout()for cluster in pipeline.estimator.micro_detector.clustered_events(events):    strongest = max(cluster, key=lambda e: e.peak_z)    print(f"t={strongest.onset - t0:6.2f}s  {strongest.duration * 1000:4.0f} ms  "          f"{len(cluster):2d} regions  strongest: {strongest.region} (z={strongest.peak_z:.1f})")

## 5. Temporal spectrumA 40–200 ms event has most of its energy in the 2–25 Hz band.  Slow head drift andlighting changes sit below it, sensor noise above.  A region with a *high* band ratiois producing short bursts; a low ratio means it is just drifting.The frame rate limits what is observable at all: at 30 fps the Nyquist frequency is15 Hz, so a 40 ms event is only 1–2 frames long.  **60 fps or more is worth having.**

In [ ]:
signals = pipeline.estimator.micro_detector.signalsratios = {name: band_energy_ratio(signal.times, signal.values, MICRO_BAND_HZ)          for name, signal in signals.items() if len(signal) >= 8}fig, ax = plt.subplots(figsize=(11, 4))names = sorted(ratios, key=ratios.get)ax.barh(names, [ratios[n] for n in names])ax.set(xlabel=f"share of spectral energy in {MICRO_BAND_HZ[0]}-{MICRO_BAND_HZ[1]} Hz",       title="Micro-motion band ratio per region", xlim=(0, 1))ax.tick_params(axis="y", labelsize=8)plt.tight_layout()fps = len(records) / max(t[-1], 1e-6)print(f"effective frame rate {fps:.1f} fps -> Nyquist {fps / 2:.1f} Hz")print(f"shortest resolvable event: ~{2000 / fps:.0f} ms")

## 6. The personal baselineEvery AU is scored as a deviation from *this subject's* neutral face.  Absolutelandmark distances differ far more between people than between expressions, so apopulation baseline would be meaningless.`sigma` is a robust (MAD-based) spread with a floor.  A `sigma` sitting exactly onits floor means the subject held that feature perfectly still during calibration —the floor is what stops that from turning noise into huge z-scores.

In [ ]:
baseline = pipeline.estimator.baselineprint(f"calibrated: {baseline.ready}\n")print(f"{'feature':<24}{'median':>10}{'sigma':>10}{'sigma/median':>14}")for name in sorted(baseline._median):    median, sigma = baseline.median(name), baseline.sigma(name)    ratio = sigma / abs(median) if abs(median) > 1e-9 else float("nan")    print(f"{name:<24}{median:10.4f}{sigma:10.4f}{ratio:14.3f}")

## 7. Where to take this next**Validate before believing.**  The AU rules here encode published FACS descriptions;none of the thresholds has been fitted to labelled data.  The honest next step is alabelled corpus:| corpus | samples | labels | note ||---|---|---|---|| CASME II | 247 | AUs + emotion, 200 fps | request from the authors || SAMM | 159 | AUs + emotion, 200 fps | request from the authors || SMIC | 164 | 3 classes | no AU labels || MAHNOB-HCI | 27 subjects | video + ECG/GSR | lets the *stress* label be checked against physiology |Concrete experiments, in order of value:1. **AU detection F1 on CASME II / SAMM.**  Feed each clip through `StressPipeline`   with the clip's own onset frames as the baseline, and compare the AUs that fire   against the coded ones.  This measures the rule set directly.2. **Threshold calibration.**  `evidence_to_intensity`'s onset/saturation and the   `min_motion` gate are currently set from the noise floor of a static clip; a   labelled corpus turns them into fitted parameters with a reported operating point.3. **Stress construct validity.**  MAHNOB-HCI has synchronised physiology; correlate   the 0–10 index against skin conductance rather than against a facial label.4. **Learned AU classifier.**  Only once (1) exists: the region flow fields are a   natural input to a small CNN, and the rule-based score becomes the baseline it   has to beat.